In [10]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [20]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [21]:
import json


def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [28]:
dataset = generate_dataset()

dataset

#with open("dataset.json", "w") as f:
    #json.dump(dataset, f, indent=2)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URI in the format 's3://bucket-name/key'. The function should return the region by querying the bucket's location constraint, or return 'us-east-1' if the query fails."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing a principal to perform only GetObject and PutObject actions on S3 buckets with names starting with 'data-'. Include appropriate Resource ARNs."},
 {'task': "Write a regular expression that validates AWS EC2 security group IDs. Security group IDs follow the pattern 'sg-' followed by exactly 8 or 17 hexadecimal characters."}]